# cyp51A Gene SNP Analysis and Genomic Neighborhood Visualization

This notebook analyzes variants in the cyp51A gene (Afu4g06890) in Aspergillus fumigatus.
We will create a heatmap visualization showing the genomic neighborhood with local coordinates.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle
import re
import warnings
warnings.filterwarnings('ignore')

## Load GTF Data

First, we load the GTF annotation file (dataset #4) to find the cyp51A gene (Afu4g06890).

In [ ]:
# Load GTF file - adjust path based on Galaxy dataset location
# In Galaxy, datasets are available in the current directory
gtf_file = 'dataset_4.dat'  # GTF annotation file from Galaxy history

# Read GTF file
gtf_columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']

try:
    gtf_data = pd.read_csv(gtf_file, sep='\t', comment='#', names=gtf_columns, dtype=str)
    print(f"Loaded GTF file with {len(gtf_data)} entries")
    print(f"Columns: {gtf_data.columns.tolist()}")
    print("\nFirst few entries:")
    print(gtf_data.head())
except Exception as e:
    print(f"Error loading GTF file: {e}")
    print("Available files in current directory:")
    import os
    print(os.listdir('.'))

## Find cyp51A Gene (Afu4g06890)

Search for the cyp51A gene using its identifier Afu4g06890 in the GTF attributes.

In [ ]:
# Function to extract gene ID from GTF attributes
def extract_gene_id(attribute_string):
    """Extract gene ID from GTF attribute string"""
    if pd.isna(attribute_string):
        return None
    
    # Look for gene_id or ID field
    patterns = [
        r'gene_id\s*"?([^\s;"]+)',
        r'ID=([^\s;]+)',
        r'Name=([^\s;]+)',
        r'locus_tag\s*"?([^\s;"]+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, str(attribute_string), re.IGNORECASE)
        if match:
            return match.group(1)
    return None

# Extract gene IDs
if 'gtf_data' in locals():
    gtf_data['gene_id'] = gtf_data['attribute'].apply(extract_gene_id)
    
    # Search for cyp51A gene (Afu4g06890)
    cyp51a_entries = gtf_data[
        gtf_data['attribute'].str.contains('Afu4g06890', case=False, na=False) |
        gtf_data['gene_id'].str.contains('Afu4g06890', case=False, na=False)
    ]
    
    print(f"Found {len(cyp51a_entries)} entries for cyp51A (Afu4g06890)")
    if len(cyp51a_entries) > 0:
        print("\ncyp51A gene entries:")
        print(cyp51a_entries[['seqname', 'feature', 'start', 'end', 'strand', 'attribute']].head(10))
    else:
        print("\nSearching for any entries containing 'cyp51' or similar:")
        cyp_entries = gtf_data[
            gtf_data['attribute'].str.contains('cyp51|CYP51', case=False, na=False)
        ]
        print(f"Found {len(cyp_entries)} entries containing 'cyp51'")
        if len(cyp_entries) > 0:
            print(cyp_entries[['seqname', 'feature', 'start', 'end', 'strand', 'attribute']].head())

## Extract Gene Structure and Genomic Coordinates

Once we find the cyp51A gene, extract its genomic coordinates, exons, and coding regions.

In [ ]:
if 'cyp51a_entries' in locals() and len(cyp51a_entries) > 0:
    # Get chromosome and coordinates
    chromosome = cyp51a_entries.iloc[0]['seqname']
    strand = cyp51a_entries.iloc[0]['strand']
    
    # Get gene boundaries
    gene_entries = cyp51a_entries[cyp51a_entries['feature'] == 'gene']
    if len(gene_entries) == 0:
        # If no gene entry, use mRNA or CDS boundaries
        gene_start = int(cyp51a_entries['start'].astype(int).min())
        gene_end = int(cyp51a_entries['end'].astype(int).max())
    else:
        gene_start = int(gene_entries.iloc[0]['start'])
        gene_end = int(gene_entries.iloc[0]['end'])
    
    print(f"cyp51A gene location:")
    print(f"Chromosome: {chromosome}")
    print(f"Coordinates: {gene_start:,} - {gene_end:,}")
    print(f"Length: {gene_end - gene_start + 1:,} bp")
    print(f"Strand: {strand}")
    
    # Extract CDS (coding sequences) for local coordinate mapping
    cds_entries = cyp51a_entries[cyp51a_entries['feature'] == 'CDS']
    if len(cds_entries) > 0:
        print(f"\nFound {len(cds_entries)} CDS entries:")
        cds_coords = []
        for _, cds in cds_entries.iterrows():
            start, end = int(cds['start']), int(cds['end'])
            cds_coords.append((start, end))
            print(f"  CDS: {start:,} - {end:,} ({end-start+1} bp)")
        
        # Sort CDS coordinates
        cds_coords.sort()
        
        # Calculate total coding length and local coordinates
        total_coding_length = sum(end - start + 1 for start, end in cds_coords)
        print(f"\nTotal coding sequence length: {total_coding_length} bp")
        
        # Create local coordinate mapping
        local_coords = {}
        local_pos = 1
        
        for start, end in cds_coords:
            for genomic_pos in range(start, end + 1):
                local_coords[genomic_pos] = local_pos
                local_pos += 1
        
        print(f"Created local coordinate mapping for {len(local_coords)} coding positions")
    
    else:
        print("No CDS entries found for cyp51A")
        cds_coords = []
        local_coords = {}
        total_coding_length = 0

else:
    print("cyp51A gene not found in GTF data")
    chromosome = None
    gene_start = gene_end = 0
    cds_coords = []
    local_coords = {}
    total_coding_length = 0

## Create Genomic Neighborhood Heatmap

Create a heatmap visualization showing:
- X-axis: genomic coordinates 
- Individual squares representing nucleotides in coding regions
- Local coordinates starting from first nucleotide of coding region

In [ ]:
if chromosome and len(cds_coords) > 0:
    # Define genomic region to visualize (gene +/- flanking region)
    flank_size = 2000  # 2kb flanking regions
    viz_start = max(1, gene_start - flank_size)
    viz_end = gene_end + flank_size
    
    print(f"\nCreating visualization for region {viz_start:,} - {viz_end:,}")
    print(f"Total region size: {viz_end - viz_start + 1:,} bp")
    
    # Create figure with subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), height_ratios=[1, 3])
    
    # Top panel: Gene structure overview
    ax1.set_xlim(viz_start, viz_end)
    ax1.set_ylim(-1, 2)
    
    # Draw gene body
    gene_rect = Rectangle((gene_start, 0), gene_end - gene_start, 0.5, 
                         facecolor='lightblue', edgecolor='blue', alpha=0.7)
    ax1.add_patch(gene_rect)
    
    # Draw CDS regions
    for start, end in cds_coords:
        cds_rect = Rectangle((start, 0), end - start, 0.5, 
                            facecolor='red', edgecolor='darkred', alpha=0.8)
        ax1.add_patch(cds_rect)
    
    ax1.set_title(f'cyp51A Gene Structure (Afu4g06890) on {chromosome}', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Gene\nStructure')
    ax1.set_xticks([])
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='lightblue', alpha=0.7, label='Gene body'),
        Patch(facecolor='red', alpha=0.8, label='Coding sequence (CDS)')
    ]
    ax1.legend(handles=legend_elements, loc='upper right')
    
    # Bottom panel: Nucleotide-level heatmap for coding regions
    if total_coding_length > 0:
        # Create matrix for coding nucleotides
        # Rows represent different aspects (sequence, local coordinates, etc.)
        # Columns represent local nucleotide positions
        
        matrix_rows = 5  # Different data tracks
        matrix = np.zeros((matrix_rows, total_coding_length))
        
        # Fill matrix with data
        col_idx = 0
        genomic_positions = []
        local_positions = []
        
        for start, end in cds_coords:
            for genomic_pos in range(start, end + 1):
                local_pos = local_coords[genomic_pos]
                genomic_positions.append(genomic_pos)
                local_positions.append(local_pos)
                
                # Row 0: Position within gene (normalized)
                matrix[0, col_idx] = (genomic_pos - gene_start) / (gene_end - gene_start)
                
                # Row 1: Local coordinate position (normalized)
                matrix[1, col_idx] = local_pos / total_coding_length
                
                # Row 2: Distance from start codon
                matrix[2, col_idx] = local_pos
                
                # Row 3: Codon position (1, 2, or 3)
                matrix[3, col_idx] = ((local_pos - 1) % 3) + 1
                
                # Row 4: Exon number
                exon_num = next(i for i, (s, e) in enumerate(cds_coords, 1) if s <= genomic_pos <= e)
                matrix[4, col_idx] = exon_num
                
                col_idx += 1
        
        # Create heatmap
        im = ax2.imshow(matrix, aspect='auto', cmap='viridis', interpolation='nearest')
        
        # Set labels and ticks
        row_labels = ['Gene Position', 'Local Coord', 'Distance from Start', 'Codon Position', 'Exon Number']
        ax2.set_yticks(range(matrix_rows))
        ax2.set_yticklabels(row_labels)
        
        # X-axis: show local coordinates
        n_ticks = min(20, total_coding_length // 50)  # Reasonable number of ticks
        if n_ticks > 0:
            tick_positions = np.linspace(0, total_coding_length-1, n_ticks, dtype=int)
            tick_labels = [str(local_positions[pos]) for pos in tick_positions]
            ax2.set_xticks(tick_positions)
            ax2.set_xticklabels(tick_labels, rotation=45)
        
        ax2.set_xlabel('Local Coding Coordinate (nucleotide position)')
        ax2.set_title('Coding Region Nucleotide-Level Analysis', fontsize=12)
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax2, shrink=0.8)
        cbar.set_label('Normalized Value', rotation=270, labelpad=20)
        
        print(f"\nHeatmap created with {total_coding_length} coding nucleotides")
        print(f"Local coordinates range from 1 to {total_coding_length}")
        
    else:
        ax2.text(0.5, 0.5, 'No coding sequence data available', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=16)
        ax2.set_xticks([])
        ax2.set_yticks([])
    
    plt.tight_layout()
    plt.savefig('cyp51A_genomic_neighborhood.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Summary statistics
    print(f"\n=== cyp51A Gene Summary ===")
    print(f"Gene ID: Afu4g06890 (cyp51A)")
    print(f"Location: {chromosome}:{gene_start:,}-{gene_end:,}")
    print(f"Gene length: {gene_end - gene_start + 1:,} bp")
    print(f"Number of exons: {len(cds_coords)}")
    print(f"Total coding length: {total_coding_length} bp")
    print(f"Strand: {strand}")

else:
    print("Cannot create visualization - cyp51A gene data not available")

## Export Gene Information

Export the cyp51A gene coordinates and local coordinate mapping for further analysis.

In [ ]:
if chromosome and len(cds_coords) > 0:
    # Create DataFrame with genomic and local coordinates
    coord_data = []
    
    for genomic_pos, local_pos in local_coords.items():
        # Find which exon this position belongs to
        exon_num = next(i for i, (s, e) in enumerate(cds_coords, 1) if s <= genomic_pos <= e)
        codon_pos = ((local_pos - 1) % 3) + 1
        
        coord_data.append({
            'chromosome': chromosome,
            'genomic_position': genomic_pos,
            'local_position': local_pos,
            'exon_number': exon_num,
            'codon_position': codon_pos,
            'gene_id': 'Afu4g06890',
            'gene_name': 'cyp51A'
        })
    
    coord_df = pd.DataFrame(coord_data)
    
    # Save coordinate mapping
    coord_df.to_csv('cyp51A_coordinate_mapping.csv', index=False)
    print(f"\nSaved coordinate mapping to 'cyp51A_coordinate_mapping.csv'")
    print(f"Contains {len(coord_df)} coding nucleotide positions")
    
    # Display first and last few positions
    print("\nFirst 10 coding positions:")
    print(coord_df.head(10))
    
    print("\nLast 10 coding positions:")
    print(coord_df.tail(10))
    
    # Gene structure summary
    structure_data = []
    for i, (start, end) in enumerate(cds_coords, 1):
        structure_data.append({
            'feature': 'CDS',
            'exon_number': i,
            'genomic_start': start,
            'genomic_end': end,
            'length_bp': end - start + 1,
            'local_start': min(pos for pos, coord in local_coords.items() if start <= pos <= end),
            'local_end': max(pos for pos, coord in local_coords.items() if start <= pos <= end)
        })
    
    structure_df = pd.DataFrame(structure_data)
    structure_df.to_csv('cyp51A_gene_structure.csv', index=False)
    print(f"\nSaved gene structure to 'cyp51A_gene_structure.csv'")
    print(structure_df)

else:
    print("No coordinate data to export")